### Optimization Problem (P) - Minimum Potential Energy Formulation

$$
\begin{aligned}
(\mathscr{P}) \quad \min_{v \in U_{ad}} \quad & J(y, v) = \frac{1}{2} \|y - z_d\|_{L^2(\Omega)}^2 + \frac{V}{2} \|v\|_{L^2(\Omega)}^2 \\
\text{subject to} \quad & y = \argmin_{w \in K} \mathscr{E}(w; v) \\
& K = \{ w \in H_0^1(\Omega) \mid w \geq 0 \text{ a.e. in } \Omega \}
\end{aligned}
$$

#### Potential Energy Functional
The energy functional $\mathscr{E}(w; v)$ is given by:

$$
\mathscr{E}(w; v) = \frac{1}{2}a(w,w) - \langle f + v, w \rangle
$$

where:
- $a(w,w) = \int_\Omega \nabla w \cdot \nabla w \, dx$ (Dirichlet energy)
- $\langle f + v, w \rangle = \int_\Omega (f + v)w \, dx$ (work done by external forces)

In [ ]:
from pathlib import Path
import sys

PACKAGE_ROOT = Path.cwd().resolve()
if not (PACKAGE_ROOT / "utils.py").is_file():
    raise FileNotFoundError("Run this notebook from the final_code package directory.")
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import matplotlib.pyplot as plt
import torch
from   torch.autograd import grad
from utils import *
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader,TensorDataset
from scipy.stats.qmc import LatinHypercube
from functools import partial
from torch.optim.lr_scheduler import LambdaLR
import os
import time 
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import clear_output
from datetime import datetime
ROOT = PACKAGE_ROOT / "outputs" / "example4"

In [ ]:
class net_y_res(nn.Module):
    def __init__(self, source, hidden=64, layers=3, init=True, activation='Tanh', hard_=None, obstacle=None):
        super(net_y_res, self).__init__()
        
        # Choose activation function based on activation_flag
        self.activation = activation
        if activation == 'Tanh':
            self.activation = nn.Tanh()
        elif activation == 'Identity':
            self.activation = nn.Identity()
        elif activation == 'Square':
            self.activation = SquaredReLU()
        elif activation == 'Cubic':
            self.activation = CubicReLU()
        elif activation == 'SELU':
            self.activation = nn.SELU()
        elif activation == 'Swish':
            self.activation = nn.SiLU()
        elif activation == 'ReLU':
            self.activation = nn.ReLU()
        
        
        # Define padding size and apply zero-padding to inputs
        self.padding_size = hidden - 2  # Padding size is hidden - 2 based on your code
        
        # Residual blocks
        self.res_blocks = nn.ModuleList(
            [ResidualBlock(hidden, self.activation) for _ in range(layers - 1)]
        )
        
        # Output layer
        self.output_layer = nn.Linear(hidden, 1)
        
        self.hard = hard_
            
        self.obstacle = obstacle  
        if init:
            self._initialize_weights()

    def _initialize_weights(self):
        # Initialize weights for the output layer
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
        
    def forward(self, x):        
        # Apply zero-padding to the input tensor
        padded_x = torch.cat([x, torch.zeros(x.size(0), self.padding_size).to(x.device)], dim=1)
        
        # Apply hard constraint (if provided)
        x1 = padded_x[:, 0:1]
        x2 = padded_x[:, 1:2]
        
        if self.hard is None:
            hard = x1 * (1 - x1) * x2 * (1 - x2)
        else:
            hard = self.hard(x1, x2)
        
        # Directly pass through residual blocks
        out = padded_x
        for res_block in self.res_blocks:
            out = res_block(out)
        
        # Output layer
        out = self.output_layer(out)
        
        # Apply hard constraint and ensure obstacle constarint
        out = out * hard
        if self.obstacle == None:
            out = torch.relu(out)
        else:
            # Apply obstacle constraint if provided
            obstcale_value = self.obstacle(x1,x2)
            out = torch.relu(out - obstcale_value) + obstcale_value
        
        return out

In [ ]:
class Source_complex:
    def __init__(self, device='cpu',nu=1.0):
        """
        Source terms for the test problem with specified y_dagger and xi_dagger
        
        Parameters:
            device: Computation device ('cpu' or 'cuda')
        """
        self.device = device
        self.nu = nu # Control weight (using standard value)

    
    def compute_f(self, x):
        """
        Compute source term f:

        """
        return 2*torch.ones_like(x[:, 0:1])  

    
    def compute_yd(self, x):
        """
        Compute desired state y_d:

        """
        return 2*torch.ones_like(x[:, 0:1])  
    


# Upper- and lower-level losses: objective and energy functional

In [ ]:
def J_u(source,net_y,net_v,x):
    """
    Objective function:
    J(y, v) = 1/2 * ||y - z_d||^2 + (V/2) * ||v||^2
    Use MSELoss to approximate the integrals.
    """
    yd = source.compute_yd
    nu = source.nu
    term1 = (1/2)*torch.mean((net_y(x)-yd(x))**2)
    term2 = (nu/2) * torch.mean(net_v(x)**2)

    return term1 +term2


def J_l(source, net_y, net_v, x):
    """
    The lower-level state y minimizes the potential energy determined by control v:
    J(w, v) = 1/2*a(w,w) - <v+f,w>
    Use MSELoss to approximate the integrals.
    """
    f = source.compute_f
    x = x.clone().detach().requires_grad_(True)
    x1 = x[:, 0:1]
    x2 = x[:, 1:2]
    x = torch.cat([x1, x2], 1)
    
    f_val = f(x)
    y_pred  = net_y(x)
    v_pred  = net_v(x)
    y_x1 = grad(y_pred.sum(), x1, create_graph=True)[0]
    y_x2 = grad(y_pred.sum(), x2, create_graph=True)[0]


    a_val = y_x1**2 + y_x2**2
    linear_term = (f_val+v_pred)*y_pred
    l = torch.mean(0.5*a_val-linear_term)
    return torch.mean(0.5*a_val-linear_term)

# Boundary functions

In [ ]:

def rho_B(theta: torch.Tensor) -> torch.Tensor:
    return 2.25 \
           + 0.21 * torch.sin(4.0 * theta) \
           + 0.18 * torch.cos(6.0 * theta) \
           + 0.135 * torch.cos(5.0 * theta)


def rho_Gamma(theta: torch.Tensor) -> torch.Tensor:
    return 1.0 \
           + 0.36 * torch.sin(3.0 * theta) \
           + 0.16 * torch.cos(2.0 * theta) 


def psi_B(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    r_2 = x**2 + y**2
    theta = torch.atan2(y, x)
    rho = rho_B(theta)
    return 1.0 - r_2 / (rho**2)

def psi_Gamma(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    r_2 = x**2 + y**2
    theta = torch.atan2(y, x)
    rho = rho_Gamma(theta)
    raw_val = 1.0 - r_2 / (rho**2)
    return 3*raw_val 

In [ ]:
def visual_y_3d(source, net_y, file_name=None):
    """
    Visualize the y solution with boundary constraints using physical coordinates directly.
    Sets values outside the domain boundary to zero and shows boundary contours at axis level.
    """
    device = next(net_y.parameters()).device
    n = 200  # Resolution
    
    # Calculate maximum radius for domain bounds
    theta_test = torch.linspace(0, 2*torch.pi, 1000, device=device)
    r_max = torch.max(rho_B(theta_test)).item()+0.1
    
    # Create grid in physical coordinates directly
    physical_coords = torch.linspace(-r_max, r_max, n, device=device)
    X_orig, Y_orig = torch.meshgrid(physical_coords, physical_coords, indexing='ij')
    xy = torch.cat((X_orig.reshape(-1,1), Y_orig.reshape(-1,1)), dim=1)

    # Calculate the model prediction - no normalization needed
    with torch.no_grad():
        y_app = net_y(xy).reshape(n, n).detach().cpu()
    
    # Convert to numpy for plotting
    X_phys = X_orig.cpu().numpy()
    Y_phys = Y_orig.cpu().numpy()
    y_app_np = y_app.numpy()
    
    # Calculate radial coordinates for proper domain checking
    R = torch.sqrt(X_orig**2 + Y_orig**2)
    Theta = torch.atan2(Y_orig, X_orig)
    
    # Get radius at each angle
    Rho_B = rho_B(Theta)
    Rho_Gamma = rho_Gamma(Theta)
    
    # Calculate boundary values - points are inside when R <= Rho_B
    outer_boundary = (Rho_B - R).cpu().numpy()
    inner_boundary = (R - Rho_Gamma).cpu().numpy()
    
    # Mask values outside the domain (R > Rho_B)
    domain_mask = (R <= Rho_B).cpu().numpy()
    y_app_np_masked = np.where(domain_mask, y_app_np, 0)
    
    # Create figure
    fig = plt.figure(figsize=(12, 10))
    
    # Create 3D plot
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X_phys, Y_phys, y_app_np_masked, cmap='jet', 
                          rstride=2, cstride=2, alpha=0.8)
    
    # Set the z-min for contours
    z_min = ax.get_zlim()[0]
    
    # Add contour for outer boundary at axis level
    outer_levels = [0]
    ax.contour(X_phys, Y_phys, outer_boundary, levels=outer_levels, colors='r', 
              linewidths=2, linestyles='solid', zdir='z', offset=z_min)
    
    # Add contour for inner boundary at axis level
    inner_levels = [0]
    ax.contour(X_phys, Y_phys, inner_boundary, levels=inner_levels, colors='blue', 
              linewidths=2, linestyles='solid', zdir='z', offset=z_min)
    
    # Add legend
    red_line = plt.Line2D([0], [0], color='red', lw=2)
    blue_line = plt.Line2D([0], [0], color='blue', lw=2)
    ax.legend([red_line, blue_line], 
             ['Outer Boundary (r=ρ_B(θ))', 'Inner Boundary (r=ρ_Γ(θ))'], 
             loc='upper right')
    
    ax.set_title("Neural Network Solution y", fontsize=14)
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)
    ax.set_zlabel("y(x,y)", fontsize=12)
    
    # Add color bar
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)
    
    # Set view angle
    ax.view_init(elev=30, azim=45)
    
    # Save or show
    plt.show()

def visual_v_3d(source, net_v, file_name=None):
    """
    Visualize the v control solution with boundary constraints using physical coordinates directly.
    Sets values outside the domain boundary to zero and shows boundary contours at axis level.
    """
    device = next(net_v.parameters()).device
    n = 200  # Resolution
    
    # Calculate maximum radius for domain bounds
    theta_test = torch.linspace(0, 2*torch.pi, 1000, device=device)
    r_max = torch.max(rho_B(theta_test)).item()+0.1
    
    # Create grid in physical coordinates directly
    physical_coords = torch.linspace(-r_max, r_max, n, device=device)
    X_orig, Y_orig = torch.meshgrid(physical_coords, physical_coords, indexing='ij')
    xy = torch.cat((X_orig.reshape(-1,1), Y_orig.reshape(-1,1)), dim=1)

    # Calculate the model prediction - no normalization needed
    with torch.no_grad():
        v_app = net_v(xy).reshape(n, n).detach().cpu()
    
    # Convert to numpy for plotting
    X_phys = X_orig.cpu().numpy()
    Y_phys = Y_orig.cpu().numpy()
    v_app_np = v_app.numpy()
    
    # Calculate radial coordinates for proper domain checking
    R = torch.sqrt(X_orig**2 + Y_orig**2)
    Theta = torch.atan2(Y_orig, X_orig)
    
    # Get radius at each angle
    Rho_B = rho_B(Theta)
    Rho_Gamma = rho_Gamma(Theta)
    
    # Calculate boundary values - points are inside when R <= Rho_B
    outer_boundary = (Rho_B - R).cpu().numpy()
    inner_boundary = (R - Rho_Gamma).cpu().numpy()
    
    # Mask values outside the domain (R > Rho_B)
    domain_mask = (R <= Rho_B).cpu().numpy()
    v_app_np_masked = np.where(domain_mask, v_app_np, 0)
    
    # Create figure
    fig = plt.figure(figsize=(12, 10))
    
    # Create 3D plot
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X_phys, Y_phys, v_app_np_masked, cmap='viridis', 
                          rstride=2, cstride=2, alpha=0.8)
    
    # Set the z-min for contours
    z_min = 0
    
    # Add contour for outer boundary at axis level
    outer_levels = [0]
    ax.contour(X_phys, Y_phys, outer_boundary, levels=outer_levels, colors='r', 
              linewidths=2, linestyles='solid', zdir='z', offset=z_min)
    
    # Add contour for inner boundary at axis level
    inner_levels = [0]
    ax.contour(X_phys, Y_phys, inner_boundary, levels=inner_levels, colors='blue', 
              linewidths=2, linestyles='solid', zdir='z', offset=z_min)
    
    # Add legend
    red_line = plt.Line2D([0], [0], color='red', lw=2)
    blue_line = plt.Line2D([0], [0], color='blue', lw=2)
    ax.legend([red_line, blue_line], 
             ['Outer Boundary (r=ρ_B(θ))', 'Inner Boundary (r=ρ_Γ(θ))'], 
             loc='upper right')
    
    ax.set_title("Neural Network Control v", fontsize=14)
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)
    ax.set_zlabel("v(x,y)", fontsize=12)
    
    # Add color bar
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)
    
    # Set view angle
    ax.view_init(elev=30, azim=45)
    
    # Save or show
    plt.show()
    if file_name is not None:
        plt.savefig(file_name, bbox_inches="tight")
        plt.close(fig)  # Close the figure to free memory

        

def visual_v_2d(source, net_v, file_name=None):
    """
    Visualize the v control solution as a 2D heatmap with boundary contours.
    Sets values outside the domain boundary to zero.
    """
    device = next(net_v.parameters()).device
    n = 200  # Resolution
    
    # Calculate maximum radius for domain bounds
    theta_test = torch.linspace(0, 2*torch.pi, 1000, device=device)
    r_max = torch.max(rho_B(theta_test)).item()+0.1
    
    # Create grid in physical coordinates directly
    physical_coords = torch.linspace(-r_max, r_max, n, device=device)
    X_orig, Y_orig = torch.meshgrid(physical_coords, physical_coords, indexing='ij')
    xy = torch.cat((X_orig.reshape(-1,1), Y_orig.reshape(-1,1)), dim=1)

    # Calculate the model prediction - no normalization needed
    with torch.no_grad():
        v_app = net_v(xy).reshape(n, n).detach().cpu()
    
    # Convert to numpy for plotting
    X_phys = X_orig.cpu().numpy()
    Y_phys = Y_orig.cpu().numpy()
    v_app_np = v_app.numpy()
    
    # Calculate radial coordinates for proper domain checking
    R = torch.sqrt(X_orig**2 + Y_orig**2)
    Theta = torch.atan2(Y_orig, X_orig)
    
    # Get radius at each angle
    Rho_B = rho_B(Theta)
    Rho_Gamma = rho_Gamma(Theta)
    
    # Create boundary masks - points are inside domain when R <= Rho_B and outside inner boundary when R >= Rho_Gamma
    domain_mask = (R <= Rho_B).cpu().numpy()
    v_app_np_masked = np.where(domain_mask, v_app_np, 0)
    
    
    # Create figure with the same size as the 3D plot
    fig = plt.figure(figsize=(12, 10))
    
    # Create 2D plot with pcolormesh for better handling of masked data
    im = plt.pcolormesh(X_phys, Y_phys, v_app_np_masked, cmap='jet', shading='auto')
    
    # Generate high-resolution boundary points
    n_boundary = 2000  # Higher resolution for smooth boundaries
    theta_boundary = torch.linspace(0, 2*torch.pi, n_boundary, device=device)
    
    # Calculate outer boundary points
    r_outer = rho_B(theta_boundary)
    x_outer = r_outer * torch.cos(theta_boundary)
    y_outer = r_outer * torch.sin(theta_boundary)
    
    # Calculate inner boundary points
    r_inner = rho_Gamma(theta_boundary)
    x_inner = r_inner * torch.cos(theta_boundary)
    y_inner = r_inner * torch.sin(theta_boundary)
    
    # Plot boundaries directly
    plt.plot(x_outer.cpu().numpy(), y_outer.cpu().numpy(), 'r-', linewidth=2)
    plt.plot(x_inner.cpu().numpy(), y_inner.cpu().numpy(), 'b-', linewidth=2)
    
    # Add legend
    red_line = plt.Line2D([0], [0], color='red', lw=2)
    blue_line = plt.Line2D([0], [0], color='blue', lw=2)
    plt.legend([red_line, blue_line], 
             ['Outer Boundary (r=ρ_B(θ))', 'Inner Boundary (r=ρ_Γ(θ))'], 
             loc='upper right')
    
    plt.title("Control Variable v", fontsize=14)
    plt.xlabel("x", fontsize=12)
    plt.ylabel("y", fontsize=12)
    plt.axis('equal')
    
    # Add color bar with same parameters as in visual_y_3d
    cbar = fig.colorbar(im, shrink=0.5, aspect=10)
    cbar.set_label("v(x,y)", fontsize=12)
    
    # Save or show
    plt.show()
    if file_name is not None:
        plt.savefig(file_name, bbox_inches="tight")
        plt.close(fig)  # Close the figure to free memory


# Random sampling in the domain

In [ ]:
class PolarBoundedDataLoader:
    """
    Random data loader for the domain defined by polar boundaries, with independent samples for y/theta and v.
    """
    def __init__(self, batch_size=64, steps_per_epoch=4, device='cpu', epsilon=1e-6, normalize=False, dual_sampling=False):
        """
        Initialize the random data loader for the polar boundary.

        Parameters:
            batch_size: Number of points per batch.
            steps_per_epoch: Number of batches per epoch.
            device: Computation device.
            normalize: Whether to normalize coordinates to [0,1]×[0,1].
            dual_sampling: Whether to return independent samples for y/theta and v.
        """
        self.batch_size = batch_size
        self.steps_per_epoch = steps_per_epoch
        self.device = device
        self.current_step = 0
        self.normalize = normalize
        self.epsilon = epsilon
        self.dual_sampling = dual_sampling
        
        # Precompute boundary values for normalization, if needed.
        if normalize:
            theta_test = torch.linspace(0, 2*torch.pi, 1000, device=device)
            r_max = torch.max(rho_B(theta_test))
            self.max_x = r_max
            self.max_y = r_max
            
    def rho_B(self, theta):
        """Define the boundary functions."""
        return 2.25 \
               + 0.21 * torch.sin(4.0 * theta) \
               + 0.18 * torch.cos(6.0 * theta) \
               + 0.135 * torch.cos(5.0 * theta)
    
    def generate_points(self):
        """Generate random points inside the polar boundary."""
        theta = 2 * torch.pi * torch.rand(self.batch_size, device=self.device)
        r_boundary = self.rho_B(theta)
        r = torch.sqrt(torch.rand(self.batch_size, device=self.device)) * r_boundary
        r = torch.clamp(r, min=self.epsilon)
        x = r * torch.cos(theta)
        y = r * torch.sin(theta)
        points = torch.stack([x, y], dim=1)
        if self.normalize:
            points = points / (2 * self.max_x) + 0.5
        return points
    
    def __iter__(self):
        self.current_step = 0
        return self
    
    def __next__(self):
        if self.current_step < self.steps_per_epoch:
            self.current_step += 1
            if self.dual_sampling:
                yz_points = self.generate_points()
                u_points = self.generate_points()
                return yz_points, u_points
            single_points = self.generate_points()
            return (single_points,)
        else:
            self.current_step = 0
            raise StopIteration
    
    def __len__(self):
        return self.steps_per_epoch

# Trainer

In [ ]:
class Trainer:
    def __init__(
        self,
        v: nn.Module,
        y: nn.Module,
        theta: nn.Module,
        source: Source,
        train_loader: DataLoader,
        optimizer_v: torch.optim.Optimizer,
        optimizer_y: torch.optim.Optimizer,
        optimizer_theta: torch.optim.Optimizer,
        scheduler_v: torch.optim.lr_scheduler.LambdaLR, 
        scheduler_y: torch.optim.lr_scheduler.LambdaLR,
        scheduler_theta: torch.optim.lr_scheduler.LambdaLR,
        gamma: float = 10.0,
        p: float = 0.5,
        device: torch.device = torch.device("cpu"),
        num_epochs: int = 10000,
        weight: list = [1, 1],
        save: bool = False,
        lr: float = 1e-3,
    ):    
        """
        Args:
            v (nn.Module): Control v(x).
            y (nn.Module): State y(x).
            xi (nn.Module):  Multiplier xi(x).
            source (Source): Source instance.
            train_loader (DataLoader): DataLoader.
            optimizer (torch.optim.Optimizer): Optimizer.
            scheduler (torch.optim.lr_scheduler.LambdaLR): Learning-rate scheduler.
            gamma (float): weight of Moreau envelope.
            p (float): parameter of penalty.
            0 <= p < 1.
            device (torch.device): Training device.
            num_epochs (int): Maximum number of training epochs.
            weight (list): weight of loss.
            save (bool): Whether to save the models.
        """

        self.v = v.to(device)
        self.y = y.to(device)
        self.theta = theta.to(device)
        self.source = source
        self.train_loader = train_loader
        self.optimizer_v = optimizer_v
        self.optimizer_y = optimizer_y
        self.optimizer_theta = optimizer_theta
        self.scheduler_v = scheduler_v
        self.scheduler_y = scheduler_y
        self.scheduler_theta = scheduler_theta
        self.gamma = gamma
        self.p = p
        self.device = device
        self.num_epochs = num_epochs
        self.weight = weight
        self.save = save
        self.num = 0
        self.timestart = 0
        self.record_u = torch.zeros(1).to(self.device)
        self.record_l = torch.zeros(1).to(self.device)
        self.lr = lr
        self.file_suffix = f"{datetime.now().strftime('%Y%m%d')}cleartheta10000_fixuboundary_gamma_{self.gamma}_cycle_factor_0.8_batch_size{batch_size}_{self.num_epochs}_p_{self.p}_lr_{self.lr}_fixinterior_norestart"
        if save == True:
            os.makedirs(f'{ROOT}/{self.file_suffix}', exist_ok=True)
        if save == True:
            self.loss_csv_file = f'{ROOT}/{self.file_suffix}/loss.csv'
            with open(self.loss_csv_file, 'w') as f:
                f.write("epoch,time,J_u_loss,J_l_loss\n")
            self.norm_csv_file = f'{ROOT}/{self.file_suffix}/norm.csv'
            with open(self.norm_csv_file, 'w') as f:
                f.write("epoch,time,rel_l2_error_y,rel_l2_error_v\n")


    def diff_net(self, net_y, net_theta):
        """
        Compute the difference between two networks, considering only linear-layer weights and biases.
        """
        res = 0
        for para1, para2 in zip(net_y.parameters(), net_theta.parameters()):
            res += torch.nn.MSELoss(reduction='sum')(para1, para2)
        return res
    
    def getloss_u(self,source,net_y, net_v, x):
        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        return self.weight[0]*J_u(source, net_y, net_v, x)
    
    def getloss_l(self, source, net_y, net_v, x):

        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        

        result = self.weight[1] * J_l(source, net_y, net_v, x)
        
        
        return result
    
    def train_one_epoch(self, epoch, timestart):
        self.v.train()
        self.y.train()
        self.theta.train()

        ck = (epoch)**self.p
        inv_ck = 1.0 / ck if ck > 0 else 0.0
        for batch in self.train_loader:
            if isinstance(batch, (tuple, list)):
                if len(batch) == 2:
                    x_yz_raw, x_u_raw = batch
                else:
                    x_yz_raw = batch[0]
                    x_u_raw = batch[0]
            else:
                x_yz_raw = batch
                x_u_raw = batch

            x_yz = x_yz_raw.detach().requires_grad_(True).to(self.device)

            with torch.set_grad_enabled(True):
                theta_params = list(self.theta.parameters())
                y_params = list(self.y.parameters())
                v_params = list(self.v.parameters())
                
                # 1. update theta
                loss_theta_e = self.getloss_l(self.source, self.theta, self.v, x_yz)
                grad_theta_e = torch.autograd.grad(loss_theta_e, theta_params, retain_graph=False)
                theta_total_grads = []
                for g_e, p_theta, p_y in zip(grad_theta_e, theta_params, y_params):
                    prox_term = (p_theta.detach() - p_y.detach()) * (1.0 / self.gamma)
                    theta_total_grads.append(g_e + prox_term)
                self._apply_custom_grads(self.theta, self.optimizer_theta, theta_total_grads)



                # 2. update y and u
                loss_upper_y = self.getloss_u(self.source, self.y, self.v, x_yz)
                loss_lower_y = self.getloss_l(self.source, self.y, self.v, x_yz)
                
                h_j_y = torch.autograd.grad(loss_upper_y, y_params, retain_graph=True)
                h_e_y = torch.autograd.grad(loss_lower_y, y_params, retain_graph=False)
                
                y_total_grads = []
                for g_j, g_e, p_y, p_theta in zip(h_j_y, h_e_y, y_params, theta_params):
                    prox_term = (p_theta.detach() - p_y.detach()) * (1.0 / self.gamma)
                    y_total_grads.append(inv_ck * g_j + g_e + prox_term)

                
                self._apply_custom_grads(self.y, self.optimizer_y, y_total_grads)
                
                x_u = x_u_raw.detach().requires_grad_(True).to(self.device)
                loss_upper_u = self.getloss_u(self.source, self.y, self.v, x_u)
                loss_lower_u = self.getloss_l(self.source, self.y, self.v, x_u)
                loss_lower_theta = self.getloss_l(self.source, self.theta, self.v, x_u)
                
                h_j_u = torch.autograd.grad(loss_upper_u, v_params, retain_graph=True)
                h_e_u = torch.autograd.grad(loss_lower_u, v_params, retain_graph=True)
                h_e_u_theta = torch.autograd.grad(loss_lower_theta, v_params, retain_graph=False)
                
                v_total_grads = []
                for g_j_u, g_e_u, g_e_u_theta in zip(h_j_u, h_e_u, h_e_u_theta):
                    v_total_grads.append(inv_ck * g_j_u + g_e_u - g_e_u_theta)

                
                self._apply_custom_grads(self.v, self.optimizer_v, v_total_grads)



        self.scheduler_v.step()
        self.scheduler_y.step()
        self.scheduler_theta.step()

        if (epoch-1) % 10 == 0:
            # Create a polar grid covering the domain defined by rho_B
            n_theta = 50  # Number of angles
            n_r = 50      # Number of radii per angle
            
            # 1. Generate angles
            thetas = torch.linspace(0, 2*torch.pi, n_theta, device=self.device)
            
            # 2. Calculate boundary radius for each theta
            boundary_radii = rho_B(thetas)
            
            # 3. Calculate area of the region using polar integration
            area = torch.sum(0.5 * boundary_radii**2 * (2*torch.pi/n_theta)).item()
            
            # 4. Create grid points within the boundary
            points_list = []
            for theta, max_r in zip(thetas, boundary_radii):
                # Create radii from 0 to boundary radius
                rs = torch.linspace(0, max_r, n_r, device=self.device)
                
                # For each (theta, r) pair, calculate (x, y)
                xs = rs * torch.cos(theta)
                ys = rs * torch.sin(theta)
                
                # Store points
                for x, y in zip(xs, ys):
                    points_list.append([x, y])
            
            # Convert points to tensor
            polar_points = torch.tensor(points_list, device=self.device)
            
            # 5. Normalize points to [0, 1] x [0, 1] as expected by model
            # Calculate maximum extents
            theta_test = torch.linspace(0, 2*torch.pi, 1000, device=self.device)
            r_max = torch.max(rho_B(theta_test))
            
            # Normalize coordinates
            grid_points = polar_points / (2 * r_max) + 0.5
            grid_points.requires_grad_(True)
            
            # Calculate losses
            self.record_u = J_u(self.source, self.y, self.v, grid_points)
            self.record_l = J_l(self.source, self.y, self.v, grid_points)
            
            # Log results
            with open(self.loss_csv_file, 'a') as f:
                f.write(f"{epoch},{time.time()-timestart},{self.record_u.item()},{self.record_l.item()}\n")
            
            if epoch % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

        return self.record_u.item(), self.record_l.item()
    
    def _update_model(self, model, optimizer, loss):
        """Update the model using manually computed gradients."""
        optimizer.zero_grad()
        grad_params = torch.autograd.grad(loss, model.parameters())
        for param, g in zip(model.parameters(), grad_params):
            param.grad = g
        optimizer.step()

    def _apply_custom_grads(self, model, optimizer, grads):
        """Update model parameters using custom gradients."""
        optimizer.zero_grad()
        for param, grad in zip(model.parameters(), grads):
            param.grad = grad
        optimizer.step()
    
    def validate(self,epoch):
        self.v.eval()
        self.y.eval()
        plt.close('all')
        #rel_l2_error_y, rel_l2_error_v = rel_l2_error_yv(self.y, self.v, self.source, device=self.device)

            
        with torch.no_grad():
            if self.save == True:
                yfile = f'{ROOT}/{self.file_suffix}/y_net_{epoch}.png'
                vfile = f'{ROOT}/{self.file_suffix}/v_net_{epoch}.png'
                xifile = f'{ROOT}/{self.file_suffix}/xi_net_{epoch}.png'
            else:
                yfile = None
            
                vfile = None
                xifile = None
            visual_y_3d(self.source, self.y, file_name=yfile)
            visual_v_2d(self.source, self.v, file_name=vfile)
        

    def fit(self):
        pbar = tqdm(range(1, self.num_epochs + 2), ascii=True, dynamic_ncols=True)
        self.timestart = time.time()
        clear_output(wait=True)
        for epoch in pbar:

            J_u_loss, J_l_loss = self.train_one_epoch(epoch, self.timestart)


            pbar.set_postfix(
                J_u_loss=J_u_loss,
                J_l_loss=J_l_loss,
            )
            
            # if (epoch-1) % 500 == 0:
            #     self.validate(epoch)
            

# Device

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
source = Source_complex(device=device)
set_seed(12345)

# Hyperparameters

In [ ]:
# hyperparameters
lr = 1e-3
warmup = 0
cycle_factor=0.8
cycle_length=1000
num_epochs=20000
weight = [1, 1]
save= True
batch_size = 512

In [ ]:
def hard(x1, x2):
    """
    Use psi_B as a hard constraint: positive inside the boundary and zero on it.
    
    Parameters:
        x1: x coordinate.
        x2: y coordinate.
    Returns:
        Constraint value: positive inside the boundary and zero on or outside it.
    """
    return torch.relu(psi_B(x1, x2))

## Bilevel MEHA training

In [ ]:
dataloader = PolarBoundedDataLoader(
    batch_size=batch_size,
    steps_per_epoch=4,
    device=device,
    dual_sampling=True
)


def psi_Gamma_(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    r_2 = x**2 + y**2
    theta = torch.atan2(y, x)
    rho = rho_Gamma(theta)
    raw_val = 1.0 - r_2 / (rho**2)
    out = 3 * raw_val
    return torch.nn.functional.relu(out)

y = net_y_res(source=source,hidden=16,layers=4,activation = 'Swish',hard_=hard,obstacle=psi_Gamma_).to(device)
v = net_v_res(source=source,hidden=16,layers=4,activation = 'Swish',hard_=hard).to(device)
theta = net_y_res(source=source,hidden=16,layers=4,activation = 'Swish',hard_=hard,obstacle=psi_Gamma_).to(device)
theta.load_state_dict(y.state_dict())


optimizer_y = torch.optim.Adam(y.parameters(), lr=lr)
optimizer_v = torch.optim.Adam(v.parameters(), lr=lr)
optimizer_theta = torch.optim.Adam(theta.parameters(), lr=lr)
scheduler_y = LambdaLR(optimizer_y,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))
scheduler_v = LambdaLR(optimizer_v,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor**2,cycle_length=cycle_length))
scheduler_theta = LambdaLR(optimizer_theta,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))

trainer = Trainer(
    v=v,
    y=y,
    theta=theta,
    source=source,
    train_loader=dataloader,
    optimizer_v=optimizer_v,
    optimizer_y=optimizer_y,
    optimizer_theta=optimizer_theta,
    scheduler_v=scheduler_v,
    scheduler_y=scheduler_y,
    scheduler_theta=scheduler_theta,
    gamma=50.0,
    p=0.2,
    device=device,
    num_epochs=num_epochs,
    weight=weight,
    save=save,
    lr = lr
)
trainer.fit()

In [ ]:
import os

# Save the final models in the models directory alongside loss.csv and norm.csv.
model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
os.makedirs(model_dir, exist_ok=True)
torch.save(y.state_dict(), os.path.join(model_dir, "final_y.pth"))
torch.save(v.state_dict(), os.path.join(model_dir, "final_v.pth"))
torch.save(theta.state_dict(), os.path.join(model_dir, "final_theta.pth"))

## After bilevel training, fix v and solve the lower-level energy minimization separately to obtain an accurate y

In [ ]:
def clone_trainer_and_train_fixed_v(original_trainer, additional_epochs=5000):
    """
    Create a clone of the trainer and continue training with v fixed, optimizing y using J_l loss.
    
    Args:
        original_trainer: The original trained Trainer instance
        additional_epochs: Number of additional epochs for the fixed-v training
        
    Returns:
        cloned_trainer: The cloned trainer instance after additional training
    """
    import copy
    import torch
    from tqdm import tqdm
    import time
    
    # Create a shallow copy of the trainer
    cloned_trainer = copy.copy(original_trainer)
    
    # Deep copy the networks (important to not affect original networks)
    cloned_trainer.y = copy.deepcopy(original_trainer.y)
    cloned_trainer.v = copy.deepcopy(original_trainer.v)
    cloned_trainer.theta = copy.deepcopy(original_trainer.theta) if original_trainer.theta is not None else None
    
    # Create fresh optimizers with the same parameters
    lr_y = original_trainer.optimizer_y.param_groups[0]['lr']
    cloned_trainer.optimizer_y = torch.optim.Adam(cloned_trainer.y.parameters(), lr=lr_y*0.1)
    
    # Copy scheduler configuration and create new scheduler
    if hasattr(original_trainer, 'scheduler_y') and original_trainer.scheduler_y is not None:
        # Get the current state of the scheduler
        from functools import partial
        from torch.optim.lr_scheduler import LambdaLR
        
        if hasattr(original_trainer, 'lr') and original_trainer.lr is not None:
            cloned_trainer.scheduler_y = LambdaLR(
                cloned_trainer.optimizer_y, 
                partial(lr_lambda, warmup=0, cycle_factor=0.8, cycle_length=1000)
            )
    
    # Set v to eval mode and freeze parameters
    cloned_trainer.v.eval()
    for param in cloned_trainer.v.parameters():
        param.requires_grad = False
    
    # Create a new file suffix to avoid overwriting original logs
    cloned_trainer.file_suffix = f"{original_trainer.file_suffix}_fixed_v_jl"
    
    # Start training loop with fixed v
    print(f"\nStarting additional training with fixed v for {additional_epochs} epochs...")
    print("Optimizing only J_l (energy functional) with fixed control")
    
    # Setup progress bar - now starting from 1
    pbar = tqdm(range(1, additional_epochs + 2), ascii=True, dynamic_ncols=True)
    
    # Training loop
    for epoch in pbar:
        # Set y to training mode
        cloned_trainer.y.train()
        
        # Train for one epoch
        total_loss = 0.0
        batch_count = 0
        
        for batch in cloned_trainer.train_loader:
            x = batch[0].detach().requires_grad_(True).to(cloned_trainer.device)
            
            # Only optimize for J_l loss
            loss_y = cloned_trainer.getloss_l(cloned_trainer.source, cloned_trainer.y, cloned_trainer.v, x)
            
            # Update y network
            cloned_trainer._update_model(cloned_trainer.y, cloned_trainer.optimizer_y, loss_y)
            
            total_loss += loss_y.item()
            batch_count += 1
        
        # Update learning rate
        if hasattr(cloned_trainer, 'scheduler_y') and cloned_trainer.scheduler_y is not None:
            cloned_trainer.scheduler_y.step()
        
        # Calculate average loss
        avg_loss = total_loss / batch_count if batch_count > 0 else 0
        
        # Record metrics periodically (without saving)
        if (epoch - 1) % 10 == 0:
            # Evaluate on grid for consistent metrics
            grid_size = 50
            x1 = torch.linspace(0, 1, grid_size, device=cloned_trainer.device)
            x2 = torch.linspace(0, 1, grid_size, device=cloned_trainer.device)
            X1, X2 = torch.meshgrid(x1, x2, indexing='ij')
            grid_points = torch.stack([X1.flatten(), X2.flatten()], dim=1).requires_grad_(True)
            
            
            # Clear cache periodically
            if epoch % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Update progress bar
        pbar.set_postfix(
            J_l_loss=avg_loss
        )
        
        # Validate periodically (just calculate error, no plotting)
        if (epoch-1) % 5000 == 0:
            cloned_trainer.y.eval()
            
            visual_y_3d(cloned_trainer.source, cloned_trainer.y, file_name=None)
            visual_v_2d(cloned_trainer.source, cloned_trainer.v, file_name=None)
            # Append results to CSV
            
            
    
    # Update the total epochs
    cloned_trainer.num_epochs = additional_epochs
    

    
    return cloned_trainer

In [ ]:
cloned_trainer = clone_trainer_and_train_fixed_v(trainer, additional_epochs=10000)

In [ ]:
import os

# Save the final models in the models directory alongside loss.csv and norm.csv.
model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
os.makedirs(model_dir, exist_ok=True)
torch.save(cloned_trainer.y.state_dict(), os.path.join(model_dir, "final_y_fixu.pth"))
torch.save(cloned_trainer.v.state_dict(), os.path.join(model_dir, "final_u_fixu.pth"))
torch.save(cloned_trainer.theta.state_dict(), os.path.join(model_dir, "final_theta_fixu.pth"))

# Plots, with the control in 3D

In [ ]:
def plot_solutions(net_y, net_v, source, device='cuda', save_path=None):
    """
    Visualize the neural network solutions for y and v with proper domain masking
    and boundary visualization, using polar coordinates to find contact boundary.
    Both y and v use 3D plotting for consistency, with v also shown as 2D projection at z=1.
    Save the output as PDF.
    """
    # Determine save directory (accept path or directory)
    save_dir = None
    if save_path:
        if os.path.isdir(save_path):
            save_dir = save_path
        else:
            maybe_dir = os.path.dirname(save_path)
            save_dir = maybe_dir if maybe_dir else save_path
        os.makedirs(save_dir, exist_ok=True)

    def tick_14(ax):
        for axis in ['x', 'y', 'z']:
            ax.tick_params(axis=axis, which='major', labelsize=14)
    
    # Calculate resolution and domain boundaries
    n = 200  # Resolution
    
    # Calculate maximum radius for domain bounds
    theta_test = torch.linspace(0, 2*torch.pi, 1000, device=device)
    r_max = torch.max(rho_B(theta_test)).item() + 0.1
    
    # Create grid in physical coordinates directly
    physical_coords = torch.linspace(-r_max, r_max, n, device=device)
    X_orig, Y_orig = torch.meshgrid(physical_coords, physical_coords, indexing='ij')
    xy = torch.cat((X_orig.reshape(-1,1), Y_orig.reshape(-1,1)), dim=1)

    # Calculate solutions
    with torch.no_grad():
        y_pred = net_y(xy).reshape(n, n).detach().cpu()
        v_pred = net_v(xy).reshape(n, n).detach().cpu()
        
        # Calculate internal obstacle values for contact boundary
        obstacle_vals = psi_Gamma(X_orig, Y_orig).detach().cpu()
    
    # Convert to numpy for plotting
    X_phys = X_orig.cpu().numpy()
    Y_phys = Y_orig.cpu().numpy()
    y_pred_np = y_pred.numpy()
    v_pred_np = v_pred.numpy()
    obstacle_np = obstacle_vals.numpy()
    
    # Calculate radial coordinates for proper domain checking
    R = torch.sqrt(X_orig**2 + Y_orig**2)
    Theta = torch.atan2(Y_orig, X_orig)
    
    # Get radius at each angle
    Rho_B = rho_B(Theta)
    Rho_Gamma = rho_Gamma(Theta)
    
    # Mask values outside the domain (R > Rho_B) - only mask outside outer boundary
    domain_mask = (R <= Rho_B).cpu().numpy()
    y_pred_masked = np.where(domain_mask, y_pred_np, 0)
    v_pred_masked = np.where(domain_mask, v_pred_np, 0)
    
    # Display obstacle values above -10, clipped from below at -2.
    obstacle_display_mask = (obstacle_np > -10) & domain_mask
    obstacle_clipped = np.where(obstacle_display_mask, 
                               np.maximum(obstacle_np, -2),
                               np.nan)
    
    # Find contact boundary using polar coordinates - improved method
    n_angles = 1000  # Number of radial directions to check
    contact_angles = []
    contact_radii = []
    contact_heights = []
    
    for angle_idx in range(n_angles):
        angle = 2 * np.pi * angle_idx / n_angles
        max_radius = float(rho_B(torch.tensor(angle, device=device)).item())
        
        # Sample along this radial direction, starting from zero.
        n_radial_samples = 300
        radii = np.linspace(0, max_radius - 0.01, n_radial_samples)
        
        # Convert to Cartesian coordinates
        x_radial = radii * np.cos(angle)
        y_radial = radii * np.sin(angle)
        
        # Create tensor for neural network evaluation
        radial_points = torch.tensor(np.stack([x_radial, y_radial], axis=1), 
                                   dtype=torch.float32, device=device)
        
        with torch.no_grad():
            y_value_pred = net_y(radial_points).cpu().numpy().flatten()
            obstacle_value = psi_Gamma(torch.tensor(x_radial, device=device), 
                                      torch.tensor(y_radial, device=device)).cpu().numpy()

        contact_indices = np.where(y_value_pred - obstacle_value <= 1e-3)[0]
        
        # Find the outermost contact point.
        if len(contact_indices) > 0:

            max_contact_idx = contact_indices[np.argmax(radii[contact_indices])]
            contact_radius = radii[max_contact_idx]
            contact_height = obstacle_value[max_contact_idx]
            
            contact_angles.append(angle)
            contact_radii.append(contact_radius)
            contact_heights.append(contact_height)

    
    # Convert contact boundary to Cartesian coordinates
    if len(contact_angles) > 0:
        contact_angles = np.array(contact_angles)
        contact_radii = np.array(contact_radii)
        contact_heights = np.array(contact_heights)
        
        # Sort by angle for smooth curve
        sort_indices = np.argsort(contact_angles)
        contact_angles = contact_angles[sort_indices]
        contact_radii = contact_radii[sort_indices]
        contact_heights = contact_heights[sort_indices]
        
        # Convert to Cartesian
        contact_x = contact_radii * np.cos(contact_angles)
        contact_y = contact_radii * np.sin(contact_angles)
    else:
        contact_x = np.array([])
        contact_y = np.array([])
        contact_heights = np.array([])
    
    # Generate high-resolution boundary points for outer and inner boundaries
    n_boundary = 1000  # Higher resolution for smooth boundaries
    theta_boundary = torch.linspace(0, 2*torch.pi, n_boundary, device=device)
    
    # Calculate outer boundary points
    r_outer = rho_B(theta_boundary)
    x_outer = (r_outer * torch.cos(theta_boundary)).cpu().numpy()
    y_outer = (r_outer * torch.sin(theta_boundary)).cpu().numpy()
    
    # Calculate inner boundary points
    r_inner = rho_Gamma(theta_boundary)
    x_inner = (r_inner * torch.cos(theta_boundary)).cpu().numpy()
    y_inner = (r_inner * torch.sin(theta_boundary)).cpu().numpy()
    
    # 1. Plot state variable y in 3D
    print("Neural Network Solution - State y")
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot solution surface
    surf = ax.plot_surface(X_phys, Y_phys, y_pred_masked, cmap='coolwarm', 
                           rstride=2, cstride=2, alpha=0.8)
    
    # Plot obstacle surface 
    obstacle_surface = ax.plot_surface(X_phys, Y_phys, obstacle_clipped, 
                                     cmap='gray', alpha=0.6, 
                                     rstride=4, cstride=4)
    
    # Set z-axis limits to start from -2
    z_max = max(np.nanmax(y_pred_masked), np.nanmax(obstacle_clipped))
    ax.set_zlim(-2, z_max)
    
    # Set the z-min for boundary projections
    z_min = -2
    
    # Plot outer boundary as 3D line at bottom
    ax.plot(x_outer, y_outer, 0, color='red', linewidth=2, linestyle='-')
    
    # Plot contact boundary as 3D line at obstacle height
    if len(contact_x) > 0:
        # Close the curve by adding the first point at the end
        contact_x_closed = np.append(contact_x, contact_x[0])
        contact_y_closed = np.append(contact_y, contact_y[0])
        contact_heights_closed = np.append(contact_heights, contact_heights[0])
        
        ax.plot(contact_x_closed, contact_y_closed, contact_heights_closed, 
                color='green', linewidth=1, linestyle='--', zorder=10)
        print("Contact boundary plotted successfully!")
    else:
        print("Warning: No contact boundary found!")
    
    # Add legend - include obstacle surface
    red_line = plt.Line2D([0], [0], color='red', lw=2)
    green_line = plt.Line2D([0], [0], color='green', lw=3, linestyle='dashed')
    gray_patch = plt.Rectangle((0, 0), 1, 1, facecolor='gray', alpha=0.6)
    
    legend_elements = [red_line]
    legend_labels = ['Domain Boundary ']
    
    if len(contact_x) > 0:
        legend_elements.append(green_line)
        legend_labels.append('Contact Boundary')
    legend_elements.append(gray_patch)
    legend_labels.append(r'Obstacle $\psi$')
    ax.legend(legend_elements, legend_labels, loc='upper right', fontsize=18)

    
    # Add color bar for the solution 
    cbar = fig.colorbar(surf, ax=ax, shrink=1.0, aspect=10)
    cbar.ax.tick_params(labelsize=14)
    tick_14(ax)
    
    # Set view angle
    ax.view_init(elev=30, azim=45)
    
    # Save or show
    if save_dir:
        plt.savefig(os.path.join(save_dir, 'example3_state.pdf'), bbox_inches="tight")
    plt.show()
    
    # 2. Plot control variable v in 3D with 2D projection at z=1
    print("Neural Network Solution - Control v")
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot control surface
    surf_v = ax.plot_surface(X_phys, Y_phys, v_pred_masked, cmap='coolwarm', 
                             rstride=2, cstride=2, alpha=0.8)
    
    # Find the z-range for proper positioning of 2D projection
    v_min = np.nanmin(v_pred_masked)
    v_max = np.nanmax(v_pred_masked)
    projection_z = v_max + 0.2 * (v_max - v_min)  # Place projection slightly above the surface
    
    # Add 2D contour projection of control variable at z=projection_z
    # Create a contour plot for better visualization
    ax.contourf(X_phys, Y_phys, v_pred_masked, 
                levels=20, cmap='coolwarm', alpha=0.6, 
                zdir='z', offset=projection_z)
    
    # Plot boundaries at the projection level
    ax.plot(x_outer, y_outer, projection_z, 'r-', linewidth=2, alpha=0.8, label='Domain Boundary')
    
    # Plot contact boundary at the projection level  
    if len(contact_x) > 0:
        contact_x_closed = np.append(contact_x, contact_x[0])
        contact_y_closed = np.append(contact_y, contact_y[0])
        ax.plot(contact_x_closed, contact_y_closed, projection_z, 'g--',
                linewidth=2,
                alpha=0.8,
                label='Contact Boundary',
                zorder=20) 
    
    # Plot boundaries at bottom level (z=0)  
    ax.plot(x_outer, y_outer, 0, color='red', linewidth=2, linestyle='-')
    
    # Set z-axis limits
    ax.set_zlim(v_min, projection_z + 0.1 * (v_max - v_min))
    
    # Add legend for control plot including contact boundary
    red_line = plt.Line2D([0], [0], color='red', lw=2)
    legend_elements = [red_line]
    legend_labels = ['Domain Boundary']
    
    if len(contact_x) > 0:
        green_line = plt.Line2D([0], [0], color='green', lw=2, linestyle='dashed')
        legend_elements.append(green_line)
        legend_labels.append('Contact Boundary')
    
    ax.legend(legend_elements, legend_labels, loc='upper right', fontsize=18)


    
    # Add color bar for the control solution
    cbar = fig.colorbar(surf_v, ax=ax, shrink=1.0, aspect=10)
    cbar.ax.tick_params(labelsize=14)
    tick_14(ax)
    
    # Set view angle
    ax.view_init(elev=30, azim=45)
    
    # Save or show
    if save_dir:
        plt.savefig(os.path.join(save_dir, 'example3_control.pdf'), bbox_inches="tight")
    plt.show()

In [ ]:
# Load the models and plot
import torch
import os


device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
source = Source_complex(device=device)

# Model paths (provided checkpoints with legacy example3 naming).
# Load the fixed-control checkpoints saved above.
model_base_path = os.path.join(ROOT, trainer.file_suffix, "models")
y_model_path = os.path.join(model_base_path, 'final_y_fixu.pth')
v_model_path = os.path.join(model_base_path, 'final_u_fixu.pth')


def hard(x1, x2):
    return torch.relu(psi_B(x1, x2))

y_net = net_y_res(source=source, hidden=16, layers=4, activation='Swish', hard_=hard, obstacle=psi_Gamma_).to(device)
v_net = net_v_res(source=source, hidden=16, layers=4, activation='Swish', hard_=hard).to(device)


y_net.load_state_dict(torch.load(y_model_path, map_location=device))
v_net.load_state_dict(torch.load(v_model_path, map_location=device))
print('Model loaded successfully!')


y_net.eval()
v_net.eval()


solution_dir = os.path.join(ROOT, 'loaded_model_results_example3')
os.makedirs(solution_dir, exist_ok=True)

# Save the state and control panels.
print('Plotting solution visualizations...')
plot_solutions(y_net, v_net, source, device=device, save_path=solution_dir)

print(f'Figures saved to: {solution_dir}')